# 第 4 课：Actor-Critic、TD 残差与 GAE

**学习目标**：掌握 PPO 训练前最容易出错的优势估计，尤其是 **terminated 与 truncated 不同**。

Actor-Critic 同时学习策略 $\pi_\theta(a|s)$ 和价值函数 $V_\phi(s)$。TD residual：

$$\delta_t=r_t+\gamma(1-\mathrm{terminated}_t)V_\phi(s_{t+1})-V_\phi(s_t).$$

广义优势估计（GAE）为

$$\hat A_t^{\mathrm{GAE}(\gamma,\lambda)}=\sum_{l=0}^{\infty}(\gamma\lambda)^l\delta_{t+l},$$

只在同一条 episode 内递推。在有限长度数据里从后往前计算：

$$\hat A_t=\delta_t+\gamma\lambda\mathbf1_{\text{episode continues}}\hat A_{t+1}.$$

最后，价值函数的监督目标经常使用 $\hat V_t^{\mathrm{target}}=\hat A_t+V_\phi(s_t)$（将右侧数值视作不反传的标签）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def gae(reward, value, next_value, terminated, truncated, gamma=0.99, lam=0.95):
    """时间序列完整包含 episode 结束标志；截断处仍可 bootstrap，但不跨重置递推。"""
    reward = np.asarray(reward, dtype=float)
    value = np.asarray(value, dtype=float)
    next_value = np.asarray(next_value, dtype=float)
    terminated = np.asarray(terminated, dtype=bool)
    truncated = np.asarray(truncated, dtype=bool)
    assert all(len(x) == len(reward) for x in (value, next_value, terminated, truncated))
    delta = reward + gamma * (~terminated) * next_value - value
    adv = np.zeros_like(reward)
    last = 0.0
    for t in reversed(range(len(reward))):
        episode_continues = not (terminated[t] or truncated[t])
        last = delta[t] + gamma * lam * episode_continues * last
        adv[t] = last
    return adv, adv + value

r = [1, 1, 1, 0]
v = [0.2, 0.3, 0.4, 0.6]
v_next = [0.3, 0.4, 0.6, 0.1]
terminated = [False, False, True, False]
truncated = [False, False, False, True]
advantages, targets = gae(r, v, v_next, terminated, truncated)
print('advantages:', np.round(advantages, 3))
print('value targets:', np.round(targets, 3))

## 4.1 用测试明确边界的物理含义

真正终止：`next_value` 不应被用来 bootstrap。

时间截断：例如训练时采样满 128 步，但物理系统仍然存在，可以借助 `V(s_next)` 估计未来回报；同时，因为下一条存储数据属于重置后的新 episode，递推优势不能跨过去。

In [ ]:
# 单步真正终止：A = r - V(s)
a_term, _ = gae([1.], [0.2], [999.], [True], [False])
assert np.allclose(a_term, [0.8])
# 单步时间截断：A = r + gamma * V(s_next) - V(s)
a_trunc, _ = gae([1.], [0.2], [2.], [False], [True])
assert np.allclose(a_trunc, [1 + 0.99*2 - 0.2])
# 截断后的不同 episode，不许把后一条的 GAE 递推回来
a_pair, _ = gae([1., 50.], [0., 0.], [2., 0.], [False, True], [True, False])
assert np.isclose(a_pair[0], 1 + .99*2)
print('All boundary tests passed.')

## 4.2 λ 如何调节偏差与方差

In [ ]:
lambdas = [0.0, 0.5, 0.95, 1.0]
plt.figure(figsize=(6,3))
for lam in lambdas:
    a, _ = gae(r, v, v_next, terminated, truncated, lam=lam)
    plt.plot(a, marker='o', label=f'lambda={lam}')
plt.legend(); plt.xlabel('Transition index'); plt.ylabel('Advantage'); plt.show()

### 练习

1. 证明 $\lambda=0$ 时 GAE 等于单步 TD residual。
2. 对固定状态序列比较 $\lambda=1$ 与 Monte Carlo return 的关系（注意时间截断的 bootstrap）。
3. 用 `pytest` 为 `gae` 增加三种边界测试，尤其测试 `terminated=True, truncated=False`。